# Edge Classification for Branch Congestion (Colab + Deep GNN)
This notebook implements a Deep Graph Neural Network pipeline (5 layers + Residual Connections) for predicting congestion across the branches of the IEEE 30-bus grid.

In [2]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "pandas>=2.2.3", "pandapower", "numpy", "torch", "torchvision",
    "torchaudio", "torch_geometric", "h5py", "optuna", "matplotlib"])

0

In [2]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Dataset
from torch_geometric.nn import GCNConv, GATConv, SAGEConv
from torch_geometric.loader import DataLoader
from torch.utils.data import random_split
import pandapower.networks as nw
from gcn_model import GCNEdgePredictor

1. PyTorch Geometric Dataset
El dataset mapea características físicas (cargas y P_max, P_min de generadores) y soporta N-1 Contingencies.

In [3]:
class IEEECongestionCSVDataset(Dataset):
    def __init__(self, csv_file):
        super().__init__()
        self.df = pd.read_csv(csv_file)
        
        net = nw.case30()
        
        self.load_buses = torch.tensor(net.load.bus.values, dtype=torch.long)
        self.gen_buses = torch.tensor(np.concatenate([net.ext_grid.bus.values, net.gen.bus.values]), dtype=torch.long)
        
        self.branches = []
        for idx, row in net.line.iterrows():
            self.branches.append((int(row.from_bus), int(row.to_bus)))
            
        self.branch_u = torch.tensor([b[0] for b in self.branches], dtype=torch.long)
        self.branch_v = torch.tensor([b[1] for b in self.branches], dtype=torch.long)
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx].values
        
        pd_data = torch.tensor(row[0:21], dtype=torch.float32)
        qd_data = torch.tensor(row[21:42], dtype=torch.float32)
        gen_status_data = torch.tensor(row[42:48], dtype=torch.float32)
        rmax_data = torch.tensor(row[48:54], dtype=torch.float32)
        rmin_data = torch.tensor(row[54:60], dtype=torch.float32)
        status_data = torch.tensor(row[60:101], dtype=torch.float32)
        targets = torch.tensor(row[101:], dtype=torch.float32).unsqueeze(-1)
        
        x = torch.zeros((30, 5), dtype=torch.float32)
        
        num_loads = min(len(self.load_buses), pd_data.shape[0])
        x[self.load_buses[:num_loads], 0] = pd_data[:num_loads]
        x[self.load_buses[:num_loads], 1] = qd_data[:num_loads]
        
        num_gens = min(len(self.gen_buses), gen_status_data.shape[0])
        x[self.gen_buses[:num_gens], 2] = gen_status_data[:num_gens]
        x[self.gen_buses[:num_gens], 3] = rmax_data[:num_gens]
        x[self.gen_buses[:num_gens], 4] = rmin_data[:num_gens]
        
        active_edges = []
        for branch_idx, is_active in enumerate(status_data):
            if is_active > 0.5:
                u, v = self.branches[branch_idx]
                active_edges.append([u, v])
                active_edges.append([v, u])
                
        if len(active_edges) > 0:
            edge_index = torch.tensor(active_edges, dtype=torch.long).t().contiguous()
        else:
            edge_index = torch.zeros((2, 0), dtype=torch.long) 
            
        data = Data(
            x=x,
            edge_index=edge_index,
            y=targets,
            status=status_data.unsqueeze(-1),
            num_nodes=30
        )
        return data

3. Execution & Training Loop

In [4]:
# ── Data ────────────────────────────────────────────────────────────────
csv_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "congestion_dataset_v3.csv")
dataset = IEEECongestionCSVDataset(csv_path)

train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size]
)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=256, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False)

# ── Model ───────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = GCNEdgePredictor(
    in_channels=5,
    hidden_channels=64,
    branch_u=dataset.branch_u,
    branch_v=dataset.branch_v,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
pos_weight = torch.tensor([10.0]).to(device)
criterion = nn.BCEWithLogitsLoss(reduction='none', pos_weight=pos_weight)
epochs = 30

print(f"Dataset: {len(dataset)} samples  (train={train_size}, val={val_size}, test={test_size})")
print(f"Model : {sum(p.numel() for p in model.parameters()):,} params  |  device={device}")
print(model)

Dataset: 50000 samples  (train=35000, val=7500, test=7500)
Model : 30,145 params  |  device=cpu
GCNEdgePredictor(
  (input_proj): Linear(in_features=5, out_features=64, bias=True)
  (convs): ModuleList(
    (0-4): 5 x GCNConv(64, 64)
  )
  (bns): ModuleList(
    (0-4): 5 x BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (mlp): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [ ]:
# ── Training Loop ───────────────────────────────────────────────────────
history = {"epoch": [], "train_loss": [], "val_acc": []}

for epoch in range(1, epochs + 1):
    # — Train —
    model.train()
    total_loss, n_batches = 0.0, 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        y = batch.y.view(-1, 1)
        mask = batch.status.view(-1, 1)
        loss = (criterion(out, y) * mask).sum() / (mask.sum() + 1e-8)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    avg_loss = total_loss / n_batches

    # — Validate —
    model.eval()
    correct, total_active = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            out = model(batch)
            y = batch.y.view(-1, 1)
            mask = batch.status.view(-1, 1)
            preds = (out > 0).float()
            correct += ((preds == y).float() * mask).sum().item()
            total_active += mask.sum().item()
    val_acc = correct / total_active

    history["epoch"].append(epoch)
    history["train_loss"].append(avg_loss)
    history["val_acc"].append(val_acc)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{epochs}  |  loss={avg_loss:.4f}  |  val_acc={val_acc:.4f}")

print("Training complete.")

Epoch   1/30  |  loss=0.7645  |  val_acc=0.8406


In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history["epoch"], history["train_loss"])
ax1.set(xlabel="Epoch", ylabel="Loss", title="Training Loss")
ax2.plot(history["epoch"], history["val_acc"])
ax2.set(xlabel="Epoch", ylabel="Accuracy", title="Validation Accuracy")
fig.tight_layout()
plt.show()

# ── Test-set evaluation ─────────────────────────────────────────────────
model.eval()
correct, total_active = 0, 0
tp, fp, fn = 0, 0, 0
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        out = model(batch)
        y = batch.y.view(-1, 1)
        mask = batch.status.view(-1, 1)
        preds = (out > 0).float()
        masked_preds = preds * mask
        masked_y = y * mask
        correct += ((preds == y).float() * mask).sum().item()
        total_active += mask.sum().item()
        tp += ((masked_preds == 1) & (masked_y == 1)).float().sum().item()
        fp += ((masked_preds == 1) & (masked_y == 0)).float().sum().item()
        fn += ((masked_preds == 0) & (masked_y == 1)).float().sum().item()

test_acc = correct / total_active
precision = tp / (tp + fp + 1e-8)
recall = tp / (tp + fn + 1e-8)
f1 = 2 * precision * recall / (precision + recall + 1e-8)

print(f"Test Accuracy : {test_acc:.4f}")
print(f"Precision     : {precision:.4f}")
print(f"Recall        : {recall:.4f}")
print(f"F1-score      : {f1:.4f}")

### 4. Hyperparameter Tuning (Optuna)
Optuna searches over hidden_channels, batch_size, learning rate, and dropout — training on the **train** split and evaluating on the **validation** split.

In [ ]:
import optuna

def objective(trial):
    hidden_channels = trial.suggest_categorical("hidden_channels", [32, 64, 128])
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.4)

    tune_train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    tune_val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    trial_model = GCNEdgePredictor(
        in_channels=5,
        hidden_channels=hidden_channels,
        branch_u=dataset.branch_u,
        branch_v=dataset.branch_v,
        dropout_rate=dropout_rate,
    ).to(device)

    opt = torch.optim.Adam(trial_model.parameters(), lr=lr)
    pos_w = torch.tensor([10.0]).to(device)
    crit = nn.BCEWithLogitsLoss(reduction='none', pos_weight=pos_w)

    for epoch in range(12):
        trial_model.train()
        for batch in tune_train_loader:
            batch = batch.to(device)
            opt.zero_grad()
            out = trial_model(batch)
            y = batch.y.view(-1, 1)
            mask = batch.status.view(-1, 1)
            loss = (crit(out, y) * mask).sum() / (mask.sum() + 1e-8)
            loss.backward()
            opt.step()

    trial_model.eval()
    correct, total_active = 0, 0
    with torch.no_grad():
        for batch in tune_val_loader:
            batch = batch.to(device)
            out = trial_model(batch)
            y = batch.y.view(-1, 1)
            mask = batch.status.view(-1, 1)
            preds = (out > 0).float()
            correct += ((preds == y).float() * mask).sum().item()
            total_active += mask.sum().item()

    return correct / total_active

print("Running Optuna hyperparameter search (20 trials)…")
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)
print(f"\nBest val accuracy: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")